# Fraud Detection — Training Notebook

A simple, self-contained fraud-detection example: a Feast feature store,
a scikit-learn model (no deep learning), and MLflow tracking. Five clear steps:
1. Data ingestion (retrieved from the Feature Store)
2. Feature processing
3. Data splitting
4. Model building
5. Model evaluation (+ MLflow logging)

## Setup

In [ ]:
import mlflow
import pandas as pd
import requests
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 1. Data ingestion

Feature values come from the Feast **Feature Store** (`feature_repo/`), not
generated ad hoc here. The `FeatureStore` custom resource clones
`feature_repo/`, runs `feast apply` to register the entity and feature view,
and materializes the data into its online store. This notebook retrieves
those already-materialized features via Feast's REST API - the same features
every consumer (this notebook, the KFP pipeline, a future serving endpoint)
would read, instead of each one regenerating its own copy of "the data."

Labels ("fraud") aren't a feature store concept - a feature store holds model
*inputs*. They're regenerated here from the same seed used to build the
feature store's data (`feature_repo/generate_data.py`), purely to keep this
notebook self-contained.

In [ ]:
FEATURE_NAMES = [
    "distance_from_last_transaction",
    "ratio_to_median_purchase_price",
    "used_chip",
    "used_pin_number",
    "online_order",
    "repeat_retailer",
]

# Internal service DNS for the Feast online feature server (REST API),
# deployed by the FeatureStore CR named "fraud-feature-store".
FEAST_ONLINE_SERVER_URL = "http://feast-fraud-feature-store-online.fraud-detection.svc.cluster.local:443"
NUM_ENTITIES = 5000

# Labels only - same seed as feature_repo/generate_data.py.
_, y = make_classification(
    n_samples=NUM_ENTITIES,
    n_features=len(FEATURE_NAMES),
    n_informative=4,
    n_redundant=0,
    weights=[0.97, 0.03],
    class_sep=1.2,
    random_state=42,
)

batch_size = 500
rows = []
for start in range(0, NUM_ENTITIES, batch_size):
    entity_ids = list(range(start, min(start + batch_size, NUM_ENTITIES)))
    response = requests.post(
        f"{FEAST_ONLINE_SERVER_URL}/get-online-features",
        json={
            "features": [f"fraud_features:{name}" for name in FEATURE_NAMES],
            "entities": {"transaction_id": entity_ids},
        },
        timeout=30,
    )
    response.raise_for_status()
    body = response.json()
    batch_df = pd.DataFrame(dict(zip(body["metadata"]["feature_names"], [r["values"] for r in body["results"]])))
    rows.append(batch_df)

df = pd.concat(rows, ignore_index=True)
df["fraud"] = y
print(f"Retrieved {len(df)} rows from the feature store, fraud rate: {df['fraud'].mean():.2%}")
df.head()

## 2. Feature processing

Scale features to zero mean / unit variance, same as the original RHOAI
fraud-detection tutorial. Fit the scaler on training data only (done after the
split below) to avoid leaking test-set information.

## 3. Data splitting

In [ ]:
X = df[FEATURE_NAMES]
y = df["fraud"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Train: {len(X_train)} rows, Test: {len(X_test)} rows")

## 4. Model building

A plain `LogisticRegression` — no deep learning, no GPU, no ONNX conversion.
`class_weight="balanced"` compensates for the rare fraud class.

In [ ]:
model = LogisticRegression(max_iter=200, class_weight="balanced")
model.fit(X_train_scaled, y_train)

## 5. Model evaluation

In [ ]:
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred)).plot()

## MLflow tracking

Log the run — parameters, metrics, and the model itself — to the platform's
MLflow server. `MLFLOW_TRACKING_URI` is already injected into this workbench's
environment (see the `opendatahub.io/mlflow-instance` annotation on the
workbench), so no manual tracking-server setup is needed here.

In [ ]:
mlflow.set_experiment("fraud-detection")

with mlflow.start_run(run_name="notebook-run") as run:
    mlflow.log_param("model_type", "logistic_regression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("features", ",".join(FEATURE_NAMES))
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.sklearn.log_model(model, "model")
    run_id = run.info.run_id

print(f"Logged MLflow run: {run_id}")
print("View it under RHOAI Dashboard -> Develop & train -> Experiments (MLflow)")

## Try a prediction

Score one example transaction, the same way the pipeline's serving step will.

In [ ]:
example_transaction = [[32.0, 1.9, 1, 0, 0, 1]]  # matches FEATURE_NAMES order
scaled = scaler.transform(pd.DataFrame(example_transaction, columns=FEATURE_NAMES))
probability = model.predict_proba(scaled)[0][1]
print(f"Fraud probability: {probability:.4f}")
print("Predicted fraud" if probability > 0.5 else "Predicted not fraud")